## Configuración:
* Importación Librerias
* Definición rutas
* Definición funciones auxiliares

In [ ]:
# Reload modulos automaticamente
%load_ext autoreload
%autoreload 2

from modules.gnn_models import (
    GCN2L,
    GraphSAGE2L,
    GAT2L,
    GCN3L,
    GraphSAGE3L,
    GAT3L,
    GCNSampler2L,
    GraphSAGESample2L,
    GATSample2L
)
import torch.nn as nn
import os
from modules.gnn import GNN
import torch_geometric
from torch_geometric.utils import degree
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, r2_score
)

print("PyG:", torch_geometric.__version__, " | PyTorch:", torch.__version__)
TOR_LABELS_DICT = {'P2P': 0, 'C2P': 1, 'P2C': 2}

## Aprender Embeddings

In [ ]:
from pathlib import Path

# Archivos del nuevo formato de grafo (nodos/aristas)
data_path = "/media/vale/KINGSTON/as-level-internet-graph/grafos/"
nodes_file = data_path + "/nodes_rib_20260301_0000_to_20260302_0000_enriched_tesis.csv"
edges_file = data_path + "/edges_rib_20260301_0000_to_20260302_0000_enriched_tesis.csv"

ribs_file = data_path + "/rib_20260301_0000_to_20260302_0000.txt"
if not Path(nodes_file).exists():
    raise FileNotFoundError(f"No existe el archivo de nodos: {nodes_file}")
if not Path(edges_file).exists():
    raise FileNotFoundError(f"No existe el archivo de aristas: {edges_file}")
if not Path(ribs_file).exists():
    raise FileNotFoundError(f"No existe el archivo de RIBs: {ribs_file}")

print(f"Nodos: {nodes_file}")
print(f"Aristas: {edges_file}")

## Funciones Auxiliares

In [ ]:
def compute_auc(pos_score, neg_score):
    pos_score = pos_score.squeeze(-1) if pos_score.dim() > 1 else pos_score
    neg_score = neg_score.squeeze(-1) if neg_score.dim() > 1 else neg_score
    scores = torch.cat([pos_score, neg_score]).numpy()
    labels = torch.cat([torch.ones(pos_score.shape[0]), torch.zeros(neg_score.shape[0])]).numpy()
    return roc_auc_score(labels, scores)


def compute_loss(pos_score, neg_score):
    pos_score = pos_score.squeeze(-1) if pos_score.dim() > 1 else pos_score
    neg_score = neg_score.squeeze(-1) if neg_score.dim() > 1 else neg_score
    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([torch.ones(pos_score.shape[0]), torch.zeros(neg_score.shape[0])])
    return F.binary_cross_entropy_with_logits(scores, labels)


def compute_accuracy(pos_score, neg_score, threshold=0.5):
    pos_score = pos_score.squeeze(-1) if pos_score.dim() > 1 else pos_score
    neg_score = neg_score.squeeze(-1) if neg_score.dim() > 1 else neg_score
    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([torch.ones(pos_score.shape[0]), torch.zeros(neg_score.shape[0])])
    preds = (scores > threshold).float()
    acc = (preds == labels).sum().item() / len(labels)
    print(f'Accuracy: {acc:.4f} (Threshold: {threshold:.2f})')
    return acc


def plot_roc_with_distributions(pos_score, neg_score, model_name, caso):
    pos_score = pos_score.squeeze(-1) if pos_score.dim() > 1 else pos_score
    neg_score = neg_score.squeeze(-1) if neg_score.dim() > 1 else neg_score
    scores = torch.cat([pos_score, neg_score]).numpy()
    labels = torch.cat([torch.ones(pos_score.shape[0]), torch.zeros(neg_score.shape[0])]).numpy()
    fpr, tpr, thresholds = roc_curve(labels, scores)
    auc = roc_auc_score(labels, scores)
    distances = np.sqrt((1 - tpr) ** 2 + fpr ** 2)
    optimal_idx = np.argmin(distances)
    optimal_threshold = thresholds[optimal_idx]
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].hist(pos_score.numpy(), bins=50, alpha=0.5, label='Positivos', color='skyblue')
    ax[0].hist(neg_score.numpy(), bins=50, alpha=0.5, label='Negativos', color='salmon')
    ax[0].axvline(x=optimal_threshold, color='red', linestyle='--',
                  label=f'Umbral={optimal_threshold:.3f}')
    ax[0].set_xlabel('Score'); ax[0].set_ylabel('Frecuencia')
    ax[0].set_title(f'Distribución de Scores: {model_name}'); ax[0].legend()
    ax[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC={auc:.4f})')
    ax[1].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
    ax[1].scatter([fpr[optimal_idx]], [tpr[optimal_idx]], color='red', zorder=5,
                  label='Umbral optimo')
    ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR')
    ax[1].set_title(f'Curva ROC - {caso}: {model_name}'); ax[1].legend(loc='lower right')
    plt.tight_layout()
    save_dir = data_path + f'results/EnfoquePorPartes/{caso}'
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(f'{save_dir}/roc_{model_name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    return optimal_threshold


def plot_loss_evolution(train_losses, num_epochs=None, title='Loss Evolution',
                        val_losses=None, test_losses=None):
    secondary = val_losses if val_losses is not None else test_losses
    ep = range(1, len(train_losses) + 1)
    plt.figure(figsize=(10, 4))
    plt.plot(ep, train_losses, label='Train Loss', linewidth=1.5)
    if secondary is not None:
        plt.plot(ep, secondary[:len(train_losses)], label='Val Loss', linewidth=1.5)
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(title)
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


## Casos


In [ ]:
import time

GLOBAL_CONFIG = {
    'lr':           0.0005,
    'hidden_dim':   256,
    'out_dim':      128,
    'dropout':      0.3,
    'epochs':       500,
    'patience':     80,
    'weight_decay': 5e-4,
    'label_smooth': 0.05,    # Ayuda a que el modelo no sea "demasiado seguro" de clases erróneas
}

### Caso 0: Predicción de enlaces utilizando grafo con atributos de nodos extraídos de Peerin

In [ ]:
# Modelos a entrenar - Caso 0
models = {
    'GCN': GCN3L,
    'GraphSAGE': GraphSAGE3L,
    'GAT': GAT3L,
}

# models = {
#     'GCN': GCN2L,
#     'GraphSAGE': GraphSAGE2L,
#     'GAT': GAT2L,
# }

In [ ]:
# 1. Cargar datos con el nuevo formato (node_id/asn + src_id/dst_id/relationship)
gnn = GNN(debug=True)
gnn.load_dataset(str(nodes_file), str(edges_file))


gnn.split_edges_link_prediction(0.8)


g0 = gnn.graph
print("\n[Resumen grafo cargado]")
print(f"- Nodos: {g0.num_nodes:,}")
print(f"- Aristas: {g0.num_edges:,}")
print(f"- Dimensión de features: {g0.x.shape[1]}")
print(f"- Edges data keys: {[k for k in g0.keys() if g0.is_edge_attr(k)]}")
# print(f"- Train edges: {g0.train_mask.sum().item():,}")
# print(f"- Val edges: {g0.val_mask.sum().item():,}")
# print(f"- Test edges: {g0.test_mask.sum().item():,}")
# if getattr(g0, "edge_label", None) is not None:
#     print(f"- Distribución de label (global): {Counter(g0.edge_label.tolist())}")
#     tr = g0.edge_label[g0.train_mask]
#     va = g0.edge_label[g0.val_mask]
#     te = g0.edge_label[g0.test_mask]
#     print(f"- Distribución train: {Counter(tr.tolist())}")
#     print(f"- Distribución val:   {Counter(va.tolist())}")
#     print(f"- Distribución test:  {Counter(te.tolist())}")

Encoder: GNN + Decoder: DotProduct

In [ ]:
import os
attr = 'mis_attr_marzo'
decoder = 'DotProduct'
in_feats      = gnn.graph.x.shape[1]
hidden_feats  = GLOBAL_CONFIG['hidden_dim']
out_feats     = GLOBAL_CONFIG['out_dim']
out_feats_mlp = 1

save_dir = data_path + 'results/EnfoquePorPartes/Caso0'
os.makedirs(save_dir, exist_ok=True)

for model_name in models:
    print(f'\nTraining model: {model_name} [DotProduct] ' + '-'*40)
    model = models[model_name](
        in_feats=in_feats,
        hidden_feats=hidden_feats,
        out_feats=out_feats,
        out_feats_mlp=out_feats_mlp,
    )
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=GLOBAL_CONFIG['lr'],
                                 weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False)

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_h = None
    patience_counter = 0

    t0 = time.perf_counter()
    for e in range(GLOBAL_CONFIG['epochs']):
        model.train()
        h = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
        pos_score = model.decodeDotProduct(gnn.train_pos_g.edge_index, h)
        neg_score = model.decodeDotProduct(gnn.train_neg_g.edge_index, h)
        loss = compute_loss(pos_score, neg_score)
        train_losses.append(loss.item())
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad():
            h_eval = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
            val_loss = compute_loss(
                model.decodeDotProduct(gnn.test_pos_g.edge_index, h_eval),
                model.decodeDotProduct(gnn.test_neg_g.edge_index, h_eval))
        val_losses.append(val_loss.item())
        scheduler.step(val_loss.item())

        if e % 10 == 0:
            print(f'  Epoch {e:3d}: train={loss.item():.4f}  val={val_loss.item():.4f}')

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_h = h_eval.clone()
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= GLOBAL_CONFIG['patience']:
            print(f'  Early stopping epoch {e}. Best val loss: {best_val_loss:.4f}')
            break
    tiempo_total = time.perf_counter() - t0
    print(f'Modelo {model_name} finalizado en {tiempo_total:.2f} segundos')

    plot_loss_evolution(train_losses, len(train_losses),
                        title=f'{model_name} (DotProduct): Evolucion del Loss',
                        val_losses=val_losses)

    with torch.no_grad():
        pos_score = model.decodeDotProduct(gnn.test_pos_g.edge_index, best_h)
        neg_score = model.decodeDotProduct(gnn.test_neg_g.edge_index, best_h)
        print('AUC:', compute_auc(pos_score, neg_score))
        threshold = plot_roc_with_distributions(
            pos_score, neg_score, model_name + '_DotProduct_' + attr, 'Caso0')
        compute_accuracy(pos_score, neg_score, threshold=threshold)

    torch.save(model.state_dict(),
               f'{save_dir}/model_emb_{decoder}_{model_name}_{attr}.pth')
    torch.save(best_h,
               f'{save_dir}/embeddings_ribs_{decoder}_{model_name}_{attr}.pt')
    print()

MLP

In [ ]:
# attr definido en celda anterior
decoder = 'MLP'
in_feats      = gnn.graph.x.shape[1]
hidden_feats  = GLOBAL_CONFIG['hidden_dim']
out_feats     = GLOBAL_CONFIG['out_dim']
out_feats_mlp = 1

save_dir = data_path + 'results/EnfoquePorPartes/Caso0'
os.makedirs(save_dir, exist_ok=True)

for model_name in models:
    print(f'\nTraining model: {model_name} [MLP] ' + '-'*40)
    model = models[model_name](
        in_feats=in_feats,
        hidden_feats=hidden_feats,
        out_feats=out_feats,
        out_feats_mlp=out_feats_mlp,
    )
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=GLOBAL_CONFIG['lr'],
                                 weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False)

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_h = None
    patience_counter = 0

    t0 = time.perf_counter()
    for e in range(GLOBAL_CONFIG['epochs']):
        model.train()
        h = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
        pos_score = model.decodeMLP(gnn.train_pos_g.edge_index, h)
        neg_score = model.decodeMLP(gnn.train_neg_g.edge_index, h)
        loss = compute_loss(pos_score, neg_score)
        train_losses.append(loss.item())
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad():
            h_eval = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
            val_loss = compute_loss(
                model.decodeMLP(gnn.test_pos_g.edge_index, h_eval),
                model.decodeMLP(gnn.test_neg_g.edge_index, h_eval))
        val_losses.append(val_loss.item())
        scheduler.step(val_loss.item())

        if e % 10 == 0:
            print(f'  Epoch {e:3d}: train={loss.item():.4f}  val={val_loss.item():.4f}')

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_h = h_eval.clone()
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= GLOBAL_CONFIG['patience']:
            print(f'  Early stopping epoch {e}. Best val loss: {best_val_loss:.4f}')
            break
    tiempo_total = time.perf_counter() - t0
    print(f'Modelo {model_name} finalizado en {tiempo_total:.2f} segundos')

    plot_loss_evolution(train_losses, len(train_losses),
                        title=f'{model_name} (MLP): Evolucion del Loss',
                        val_losses=val_losses)

    with torch.no_grad():
        pos_score = model.decodeMLP(gnn.test_pos_g.edge_index, best_h)
        neg_score = model.decodeMLP(gnn.test_neg_g.edge_index, best_h)
        print('AUC:', compute_auc(pos_score, neg_score))
        threshold = plot_roc_with_distributions(
            pos_score, neg_score, model_name + '_MLP_' + attr, 'Caso0')
        compute_accuracy(pos_score, neg_score, threshold=threshold)

    torch.save(model.state_dict(),
               f'{save_dir}/model_emb_{decoder}_{model_name}_{attr}.pth')
    torch.save(best_h,
               f'{save_dir}/embeddings_ribs_{decoder}_{model_name}_{attr}.pt')
    print()

### Caso 1: Predicción de enlaces utilizando grafo con atributos de centralidad

In [ ]:
models = {
    'GCN': GCN3L,
    'GraphSAGE': GraphSAGE3L,
    'GAT': GAT3L,
}

# models = {
#     'GCN': GCN2L,
#     'GraphSAGE': GraphSAGE2L,
#     'GAT': GAT2L,
# }

# CASO 1: Grafo con atributos de grado de entrada y salida:
# ---------------------------------------
attr = 'grado_attr_febrero' 


In [ ]:
# 1. Cargar datos con el nuevo formato (node_id/asn + src_id/dst_id/relationship)
gnn = GNN(debug=True)
gnn.load_dataset(str(nodes_file), str(edges_file))

gnn.split_edges_link_prediction(0.8)

g0 = gnn.graph
print("\n[Resumen grafo cargado]")
print(f"- Nodos: {g0.num_nodes:,}")
print(f"- Aristas: {g0.num_edges:,}")
print(f"- Dimensión de features: {g0.x.shape[1]}")
print(f"- Edges data keys: {[k for k in g0.keys() if g0.is_edge_attr(k)]}")

DotProduct

In [ ]:
# attr = 'grado_attr_febrero' (definido arriba)
decoder = 'DotProduct'
in_feats      = gnn.graph.x.shape[1]
hidden_feats  = GLOBAL_CONFIG['hidden_dim']
out_feats     = GLOBAL_CONFIG['out_dim']
out_feats_mlp = 1

save_dir = data_path + 'results/EnfoquePorPartes/Caso1'
os.makedirs(save_dir, exist_ok=True)

for model_name in models:
    print(f'\nTraining model: {model_name} [DotProduct] ' + '-'*40)
    model = models[model_name](
        in_feats=in_feats,
        hidden_feats=hidden_feats,
        out_feats=out_feats,
        out_feats_mlp=out_feats_mlp,
    )
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=GLOBAL_CONFIG['lr'],
                                 weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False)

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_h = None
    patience_counter = 0

    t0 = time.perf_counter()
    for e in range(GLOBAL_CONFIG['epochs']):
        model.train()
        h = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
        pos_score = model.decodeDotProduct(gnn.train_pos_g.edge_index, h)
        neg_score = model.decodeDotProduct(gnn.train_neg_g.edge_index, h)
        loss = compute_loss(pos_score, neg_score)
        train_losses.append(loss.item())
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad():
            h_eval = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
            val_loss = compute_loss(
                model.decodeDotProduct(gnn.test_pos_g.edge_index, h_eval),
                model.decodeDotProduct(gnn.test_neg_g.edge_index, h_eval))
        val_losses.append(val_loss.item())
        scheduler.step(val_loss.item())

        if e % 10 == 0:
            print(f'  Epoch {e:3d}: train={loss.item():.4f}  val={val_loss.item():.4f}')

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_h = h_eval.clone()
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= GLOBAL_CONFIG['patience']:
            print(f'  Early stopping epoch {e}. Best val loss: {best_val_loss:.4f}')
            break
    tiempo_total = time.perf_counter() - t0
    print(f'Modelo {model_name} finalizado en {tiempo_total:.2f} segundos')

    plot_loss_evolution(train_losses, len(train_losses),
                        title=f'{model_name} (DotProduct): Evolucion del Loss',
                        val_losses=val_losses)

    with torch.no_grad():
        pos_score = model.decodeDotProduct(gnn.test_pos_g.edge_index, best_h)
        neg_score = model.decodeDotProduct(gnn.test_neg_g.edge_index, best_h)
        print('AUC:', compute_auc(pos_score, neg_score))
        threshold = plot_roc_with_distributions(
            pos_score, neg_score, model_name + '_DotProduct_' + attr, 'Caso1')
        compute_accuracy(pos_score, neg_score, threshold=threshold)

    torch.save(model.state_dict(),
               f'{save_dir}/model_emb_{decoder}_{model_name}_{attr}.pth')
    torch.save(best_h,
               f'{save_dir}/embeddings_ribs_{decoder}_{model_name}_{attr}.pt')
    print()

MLP


In [ ]:
# attr = 'grado_attr_febrero' (definido arriba)
decoder = 'MLP'
in_feats      = gnn.graph.x.shape[1]
hidden_feats  = GLOBAL_CONFIG['hidden_dim']
out_feats     = GLOBAL_CONFIG['out_dim']
out_feats_mlp = 1

save_dir = data_path + 'results/EnfoquePorPartes/Caso1'
os.makedirs(save_dir, exist_ok=True)

for model_name in models:
    print(f'\nTraining model: {model_name} [MLP] ' + '-'*40)
    model = models[model_name](
        in_feats=in_feats,
        hidden_feats=hidden_feats,
        out_feats=out_feats,
        out_feats_mlp=out_feats_mlp,
    )
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=GLOBAL_CONFIG['lr'],
                                 weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False)

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_h = None
    patience_counter = 0

    t0 = time.perf_counter()
    for e in range(GLOBAL_CONFIG['epochs']):
        model.train()
        h = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
        pos_score = model.decodeMLP(gnn.train_pos_g.edge_index, h)
        neg_score = model.decodeMLP(gnn.train_neg_g.edge_index, h)
        loss = compute_loss(pos_score, neg_score)
        train_losses.append(loss.item())
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad():
            h_eval = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
            val_loss = compute_loss(
                model.decodeMLP(gnn.test_pos_g.edge_index, h_eval),
                model.decodeMLP(gnn.test_neg_g.edge_index, h_eval))
        val_losses.append(val_loss.item())
        scheduler.step(val_loss.item())

        if e % 10 == 0:
            print(f'  Epoch {e:3d}: train={loss.item():.4f}  val={val_loss.item():.4f}')

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_h = h_eval.clone()
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= GLOBAL_CONFIG['patience']:
            print(f'  Early stopping epoch {e}. Best val loss: {best_val_loss:.4f}')
            break
    tiempo_total = time.perf_counter() - t0
    print(f'Modelo {model_name} finalizado en {tiempo_total:.2f} segundos')

    plot_loss_evolution(train_losses, len(train_losses),
                        title=f'{model_name} (MLP): Evolucion del Loss',
                        val_losses=val_losses)

    with torch.no_grad():
        pos_score = model.decodeMLP(gnn.test_pos_g.edge_index, best_h)
        neg_score = model.decodeMLP(gnn.test_neg_g.edge_index, best_h)
        print('AUC:', compute_auc(pos_score, neg_score))
        threshold = plot_roc_with_distributions(
            pos_score, neg_score, model_name + '_MLP_' + attr, 'Caso1')
        compute_accuracy(pos_score, neg_score, threshold=threshold)

    torch.save(model.state_dict(),
               f'{save_dir}/model_emb_{decoder}_{model_name}_{attr}.pth')
    torch.save(best_h,
               f'{save_dir}/embeddings_ribs_{decoder}_{model_name}_{attr}.pt')
    print()

### Caso 2: Predicción de enlaces utilizando grafo con atributos provenientes de estudios pre

In [ ]:
models = {
    'GCN': GCN3L,
    'GraphSAGE': GraphSAGE3L,
    'GAT': GAT3L,
}
# models = {
#     'GCN': GCN2L,
#     'GraphSAGE': GraphSAGE2L,
#     'GAT': GAT2L,
# }

# CASO 2: Grafo 2022 con atributos de nodos del paper
# ---------------------------------------
attr = '2022_attr'
INTERNETGRAPH_FILE = '/media/vale/KINGSTON/TESIS/InternetGNNData2022/caida/caida.bin'

# caida.bin es un binario DGL; PyG no lo lee. Convertir una sola vez desde un entorno con DGL:
#   graphs, _ = dgl.load_graphs('.../caida.bin'); g = graphs[0]
#   data = Data(x=g.ndata['feat'], edge_index=torch.stack(g.edges()), num_nodes=g.num_nodes())
#   torch.save(data, '.../caida_pyg.pt')
INTERNETGRAPH_FILE = INTERNETGRAPH_FILE.replace('caida.bin', 'caida_pyg.pt')

gnn = GNN(debug=True)
gnn.graph = torch.load(INTERNETGRAPH_FILE, weights_only=False)
print(f'Grafo 2022: {gnn.graph.num_nodes:,} nodos, {gnn.graph.num_edges:,} aristas')
print(f'Dimensión de features: {gnn.graph.x.shape[1]}')

gnn.split_edges_link_prediction(0.8)


DotProduct

In [ ]:
# attr = '2022_attr' (definido arriba)
decoder = 'DotProduct'
in_feats      = gnn.graph.x.shape[1]
hidden_feats  = GLOBAL_CONFIG['hidden_dim']
out_feats     = GLOBAL_CONFIG['out_dim']
out_feats_mlp = 1

save_dir = data_path + 'results/EnfoquePorPartes/Caso2'
os.makedirs(save_dir, exist_ok=True)

for model_name in models:
    print(f'\nTraining model: {model_name} [DotProduct] ' + '-'*40)
    model = models[model_name](
        in_feats=in_feats,
        hidden_feats=hidden_feats,
        out_feats=out_feats,
        out_feats_mlp=out_feats_mlp,
    )
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=GLOBAL_CONFIG['lr'],
                                 weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False)

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_h = None
    patience_counter = 0

    t0 = time.perf_counter()
    for e in range(GLOBAL_CONFIG['epochs']):
        model.train()
        h = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
        pos_score = model.decodeDotProduct(gnn.train_pos_g.edge_index, h)
        neg_score = model.decodeDotProduct(gnn.train_neg_g.edge_index, h)
        loss = compute_loss(pos_score, neg_score)
        train_losses.append(loss.item())
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad():
            h_eval = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
            val_loss = compute_loss(
                model.decodeDotProduct(gnn.test_pos_g.edge_index, h_eval),
                model.decodeDotProduct(gnn.test_neg_g.edge_index, h_eval))
        val_losses.append(val_loss.item())
        scheduler.step(val_loss.item())

        if e % 10 == 0:
            print(f'  Epoch {e:3d}: train={loss.item():.4f}  val={val_loss.item():.4f}')

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_h = h_eval.clone()
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= GLOBAL_CONFIG['patience']:
            print(f'  Early stopping epoch {e}. Best val loss: {best_val_loss:.4f}')
            break
    tiempo_total = time.perf_counter() - t0
    print(f'Modelo {model_name} finalizado en {tiempo_total:.2f} segundos')

    plot_loss_evolution(train_losses, len(train_losses),
                        title=f'{model_name} (DotProduct): Evolucion del Loss',
                        val_losses=val_losses)

    with torch.no_grad():
        pos_score = model.decodeDotProduct(gnn.test_pos_g.edge_index, best_h)
        neg_score = model.decodeDotProduct(gnn.test_neg_g.edge_index, best_h)
        print('AUC:', compute_auc(pos_score, neg_score))
        threshold = plot_roc_with_distributions(
            pos_score, neg_score, model_name + '_DotProduct_' + attr, 'Caso2')
        compute_accuracy(pos_score, neg_score, threshold=threshold)

    torch.save(model.state_dict(),
               f'{save_dir}/model_emb_{decoder}_{model_name}_{attr}.pth')
    torch.save(best_h,
               f'{save_dir}/embeddings_ribs_{decoder}_{model_name}_{attr}.pt')
    print()

MLP

In [ ]:
# attr = '2022_attr' (definido arriba)
decoder = 'MLP'
in_feats      = gnn.graph.x.shape[1]
hidden_feats  = GLOBAL_CONFIG['hidden_dim']
out_feats     = GLOBAL_CONFIG['out_dim']
out_feats_mlp = 1

save_dir = data_path + 'results/EnfoquePorPartes/Caso2'
os.makedirs(save_dir, exist_ok=True)

for model_name in models:
    print(f'\nTraining model: {model_name} [MLP] ' + '-'*40)
    model = models[model_name](
        in_feats=in_feats,
        hidden_feats=hidden_feats,
        out_feats=out_feats,
        out_feats_mlp=out_feats_mlp,
    )
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=GLOBAL_CONFIG['lr'],
                                 weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False)

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_h = None
    patience_counter = 0

    t0 = time.perf_counter()
    for e in range(GLOBAL_CONFIG['epochs']):
        model.train()
        h = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
        pos_score = model.decodeMLP(gnn.train_pos_g.edge_index, h)
        neg_score = model.decodeMLP(gnn.train_neg_g.edge_index, h)
        loss = compute_loss(pos_score, neg_score)
        train_losses.append(loss.item())
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad():
            h_eval = model.encode(gnn.train_g.x, gnn.train_g.edge_index)
            val_loss = compute_loss(
                model.decodeMLP(gnn.test_pos_g.edge_index, h_eval),
                model.decodeMLP(gnn.test_neg_g.edge_index, h_eval))
        val_losses.append(val_loss.item())
        scheduler.step(val_loss.item())

        if e % 10 == 0:
            print(f'  Epoch {e:3d}: train={loss.item():.4f}  val={val_loss.item():.4f}')

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_h = h_eval.clone()
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= GLOBAL_CONFIG['patience']:
            print(f'  Early stopping epoch {e}. Best val loss: {best_val_loss:.4f}')
            break
    tiempo_total = time.perf_counter() - t0
    print(f'Modelo {model_name} finalizado en {tiempo_total:.2f} segundos')

    plot_loss_evolution(train_losses, len(train_losses),
                        title=f'{model_name} (MLP): Evolucion del Loss',
                        val_losses=val_losses)

    with torch.no_grad():
        pos_score = model.decodeMLP(gnn.test_pos_g.edge_index, best_h)
        neg_score = model.decodeMLP(gnn.test_neg_g.edge_index, best_h)
        print('AUC:', compute_auc(pos_score, neg_score))
        threshold = plot_roc_with_distributions(
            pos_score, neg_score, model_name + '_MLP_' + attr, 'Caso2')
        compute_accuracy(pos_score, neg_score, threshold=threshold)

    torch.save(model.state_dict(),
               f'{save_dir}/model_emb_{decoder}_{model_name}_{attr}.pth')
    torch.save(best_h,
               f'{save_dir}/embeddings_ribs_{decoder}_{model_name}_{attr}.pt')
    print()

### Caso 3: Predicción del grado de los nodos utilizando grafo con atributos extraídos de Peering

In [ ]:
models = {
    'GCN': GCN3L,
    'GraphSAGE': GraphSAGE3L,
    'GAT': GAT3L,
}

# models = {
#     'GCN': GCN2L,
#     'GraphSAGE': GraphSAGE2L,
#     'GAT': GAT2L,
# }


# CASO 3: Grafo con atributos de nodos extraídos de PeeringDB
# ---------------------------------------
attr = 'out_degree'

# 1. Cargar datos con el nuevo formato (node_id/asn + src_id/dst_id/relationship)
gnn = GNN(debug=True)
gnn.load_dataset(str(nodes_file), str(edges_file))


gnn.split_graph_nodes(train_size=0.8)


g0 = gnn.graph

In [ ]:
degrees = degree(gnn.graph.edge_index[0], gnn.graph.num_nodes).float().unsqueeze(1)

# Transformación logarítmica
deg_log = torch.log1p(degrees)  # log(1 + x)

# Min-max normalization después del log
min_val = deg_log.min()
max_val = deg_log.max()
deg_log_norm = (deg_log - min_val) / (max_val - min_val + 1e-8)

gnn.graph.out_degree = deg_log_norm



encode: GNN

In [ ]:
attr = 'out_degree'
features = gnn.graph.x
labels   = gnn.graph.out_degree

# Split: 64% train / 16% val / 20% test
train_mask_orig     = gnn.graph.node_train_mask
test_mask           = gnn.graph.node_test_mask
train_indices       = torch.where(train_mask_orig)[0]
g_seed              = torch.Generator().manual_seed(42)
perm                = torch.randperm(len(train_indices), generator=g_seed)
n_val               = int(0.2 * len(train_indices))
val_indices         = train_indices[perm[:n_val]]
train_indices_final = train_indices[perm[n_val:]]
val_mask  = torch.zeros(gnn.graph.num_nodes, dtype=torch.bool)
val_mask[val_indices] = True
train_mask = torch.zeros(gnn.graph.num_nodes, dtype=torch.bool)
train_mask[train_indices_final] = True
print(f'Split => Train: {train_mask.sum()} | Val: {val_mask.sum()} | Test: {test_mask.sum()}')

in_feats      = features.shape[1]
hidden_feats  = GLOBAL_CONFIG['hidden_dim']
out_feats     = GLOBAL_CONFIG['out_dim']
out_feats_mlp = 1
save_dir      = data_path + 'results/EnfoquePorPartes/Caso3'
os.makedirs(save_dir, exist_ok=True)

for model_name in models:
    print(f'\nTraining model: {model_name} ' + '-'*40)
    model = models[model_name](
        in_feats=in_feats,
        hidden_feats=hidden_feats,
        out_feats=out_feats,
        out_feats_mlp=out_feats_mlp,
        edge_dim=1,
    )
    loss_fn   = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=GLOBAL_CONFIG['lr'],
                                 weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False)

    train_losses, val_losses = [], []
    best_val_loss    = float('inf')
    best_state       = None
    patience_counter = 0

    t0 = time.perf_counter()
    for epoch in range(GLOBAL_CONFIG['epochs']):
        model.train()
        pred = model(features, gnn.graph.edge_index)
        loss = loss_fn(pred[train_mask], labels[train_mask])
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad():
            pred_eval = model(features, gnn.graph.edge_index)
            train_mae = loss_fn(pred_eval[train_mask], labels[train_mask]).item()
            val_mae   = loss_fn(pred_eval[val_mask],   labels[val_mask]).item()
        train_losses.append(train_mae)
        val_losses.append(val_mae)
        scheduler.step(val_mae)

        if epoch % 10 == 0 or epoch == GLOBAL_CONFIG['epochs'] - 1:
            print(f'  Epoch {epoch:3d}: train={train_mae:.6f}  val={val_mae:.6f}')

        if val_mae < best_val_loss:
            best_val_loss    = val_mae
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= GLOBAL_CONFIG['patience']:
            print(f'  Early stopping epoch {epoch}. Best val MAE: {best_val_loss:.6f}')
            break
    tiempo_total = time.perf_counter() - t0
    print(f'Modelo {model_name} finalizado en {tiempo_total:.2f} segundos')

    model.load_state_dict(best_state)
    model.eval()

    plot_loss_evolution(train_losses, len(train_losses),
                        title=f'{model_name}: Evolución del Loss (MAE) — out_degree',
                        val_losses=val_losses)

    with torch.no_grad():
        pred_final = model(features, gnn.graph.edge_index)
        embeddings = model.encode(features, gnn.graph.edge_index)

    pred_test   = pred_final[test_mask].cpu().numpy().flatten()
    labels_test = labels[test_mask].cpu().numpy().flatten()
    test_mae    = loss_fn(pred_final[test_mask], labels[test_mask]).item()
    r2          = r2_score(labels_test, pred_test)

    plt.figure(figsize=(5, 4))
    plt.scatter(labels_test, pred_test, alpha=0.15, s=4)
    plt.xlabel('Real (out_degree norm)')
    plt.ylabel('Predicho')
    plt.title(f'{model_name}: Real vs Predicted')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'\n  [Métricas {model_name}]')
    print(f'  Best Val MAE : {best_val_loss:.6f}')
    print(f'  Test MAE     : {test_mae:.6f}')
    print(f'  Test R²      : {r2:.6f}')

    torch.save(model.state_dict(),
               f'{save_dir}/model_emb_{model_name}_{attr}.pth')
    torch.save(embeddings,
               f'{save_dir}/embeddings_ribs_{model_name}_{attr}.pt')
    print(f'  Guardado: embeddings_ribs_{model_name}_{attr}.pt')
    print()

In [ ]:
# Estadísticas del grafo
num_nodes = gnn.graph.num_nodes
num_edges = gnn.graph.num_edges
in_degrees = degree(gnn.graph.edge_index[1], gnn.graph.num_nodes).long().numpy()
out_degrees = degree(gnn.graph.edge_index[0], gnn.graph.num_nodes).long().numpy()

print("=" * 60)
print("ESTADÍSTICAS DEL GRAFO")
print("=" * 60)
print(f"Número de nodos (ASes): {num_nodes:,}")
print(f"Número de aristas: {num_edges:,}")
print(f"Densidad del grafo: {num_edges / (num_nodes * (num_nodes - 1)):.6f}")
print(f"\nGrado de entrada promedio: {in_degrees.mean():.2f}")
print(f"Grado de salida promedio: {out_degrees.mean():.2f}")
print(f"Grado de entrada máximo: {in_degrees.max()}")
print(f"Grado de salida máximo: {out_degrees.max()}")

# Visualizar distribución de grados
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histograma de grados de entrada
axes[0].hist(in_degrees, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Grado de Entrada', fontsize=11)
axes[0].set_ylabel('Frecuencia', fontsize=11)
axes[0].set_title('Distribución de Grados de Entrada', fontsize=12)
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Histograma de grados de salida
axes[1].hist(out_degrees, bins=50, color='salmon', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Grado de Salida', fontsize=11)
axes[1].set_ylabel('Frecuencia', fontsize=11)
axes[1].set_title('Distribución de Grados de Salida', fontsize=12)
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

# Scatter plot: in-degree vs out-degree
axes[2].scatter(in_degrees, out_degrees, alpha=0.3, s=10)
axes[2].set_xlabel('Grado de Entrada', fontsize=11)
axes[2].set_ylabel('Grado de Salida', fontsize=11)
axes[2].set_title('Grado de Entrada vs Salida', fontsize=12)
axes[2].set_xscale('log')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Caso 4: Predicción del valor de PageRank utilizando grafo con atributos extraídos de PeeringDB

In [ ]:
models = {
    'GCN': GCN3L,
    'GraphSAGE': GraphSAGE3L,
    'GAT': GAT3L,
}
# models = {
#     'GCN': GCN2L,
#     'GraphSAGE': GraphSAGE2L,
#     'GAT': GAT2L,
# }



attr = 'pagerank'

# Cargar grafo 
# ---------------------- 
gnn = GNN(debug=True)
gnn.load_dataset(str(nodes_file), str(edges_file))


gnn.split_graph_nodes(train_size=0.8)


g4 = gnn.graph
g4


Calculo PageRank

In [ ]:
N    = gnn.graph.num_nodes
DAMP = 0.85
K    = 20   # más iteraciones para mejor convergencia

def compute_pagerank(g):
    # Message passing explícito: cada nodo envía pv/deg a sus vecinos de salida (copy_u + sum)
    src, dst = g.edge_index
    pv = torch.ones(N) / N
    degrees = degree(src, N).float()
    degrees = torch.clamp(degrees, min=1)
    for _ in range(K):
        pv = pv / degrees
        pv = torch.zeros(N).index_add_(0, dst, pv[src])
        pv = (1 - DAMP) / N + DAMP * pv
    g.pv = pv
    return g.pv   # (N,)

pv = compute_pagerank(gnn.graph)  # (N,) sin unsqueeze todavía
gnn.graph.pagerank = pv.unsqueeze(1)

# Normalización: log1p primero (aplana la cola power-law) + min-max → [0,1]
pr_log  = torch.log1p(pv)
pr_min  = pr_log.min()
pr_max  = pr_log.max()
pr_norm = (pr_log - pr_min) / (pr_max - pr_min + 1e-12)
gnn.graph.pagerank_norm = pr_norm.unsqueeze(1)

print(f'PageRank raw   — min={pv.min():.6f}  max={pv.max():.6f}  mean={pv.mean():.6f}')
print(f'PageRank norm  — min={pr_norm.min():.4f}  max={pr_norm.max():.4f}  mean={pr_norm.mean():.4f}')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pv.numpy(), bins=100, log=True, color='steelblue', alpha=0.8)
axes[0].set_title('PageRank raw (escala log)'); axes[0].set_xlabel('PageRank')
axes[1].hist(pr_norm.numpy(), bins=100, color='darkorange', alpha=0.8)
axes[1].set_title('PageRank log1p + min-max norm'); axes[1].set_xlabel('PageRank normalizado')
plt.tight_layout(); plt.show()

gnn.graph


encode: GNN

In [ ]:
attr     = 'pagerank_norm'
features = gnn.graph.x
labels   = gnn.graph.pagerank_norm

# Split: 64% train / 16% val / 20% test
train_mask_orig     = gnn.graph.node_train_mask
test_mask           = gnn.graph.node_test_mask
train_indices       = torch.where(train_mask_orig)[0]
g_seed              = torch.Generator().manual_seed(42)
perm                = torch.randperm(len(train_indices), generator=g_seed)
n_val               = int(0.2 * len(train_indices))
val_indices         = train_indices[perm[:n_val]]
train_indices_final = train_indices[perm[n_val:]]
val_mask  = torch.zeros(gnn.graph.num_nodes, dtype=torch.bool)
val_mask[val_indices] = True
train_mask = torch.zeros(gnn.graph.num_nodes, dtype=torch.bool)
train_mask[train_indices_final] = True
print(f'Split => Train: {train_mask.sum()} | Val: {val_mask.sum()} | Test: {test_mask.sum()}')

in_feats      = features.shape[1]
hidden_feats  = GLOBAL_CONFIG['hidden_dim']
out_feats     = GLOBAL_CONFIG['out_dim']
out_feats_mlp = 1
save_dir      = data_path + 'results/EnfoquePorPartes/Caso4'
os.makedirs(save_dir, exist_ok=True)

for model_name in models:
    print(f'\nTraining model: {model_name} ' + '-'*40)
    model = models[model_name](
        in_feats=in_feats,
        hidden_feats=hidden_feats,
        out_feats=out_feats,
        out_feats_mlp=out_feats_mlp,
    )
    loss_fn   = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=GLOBAL_CONFIG['lr'],
                                 weight_decay=GLOBAL_CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False)

    train_losses, val_losses = [], []
    best_val_loss    = float('inf')
    best_state       = None
    patience_counter = 0

    t0 = time.perf_counter()
    for epoch in range(GLOBAL_CONFIG['epochs']):
        model.train()
        pred = model(features, gnn.graph.edge_index)
        loss = loss_fn(pred[train_mask], labels[train_mask])
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad():
            pred_eval = model(features, gnn.graph.edge_index)
            train_mae = loss_fn(pred_eval[train_mask], labels[train_mask]).item()
            val_mae   = loss_fn(pred_eval[val_mask],   labels[val_mask]).item()
        train_losses.append(train_mae)
        val_losses.append(val_mae)
        scheduler.step(val_mae)

        if epoch % 10 == 0 or epoch == GLOBAL_CONFIG['epochs'] - 1:
            print(f'  Epoch {epoch:3d}: train={train_mae:.6f}  val={val_mae:.6f}')

        if val_mae < best_val_loss:
            best_val_loss    = val_mae
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= GLOBAL_CONFIG['patience']:
            print(f'  Early stopping epoch {epoch}. Best val MAE: {best_val_loss:.6f}')
            break
    tiempo_total = time.perf_counter() - t0
    print(f'Modelo {model_name} finalizado en {tiempo_total:.2f} segundos')

    model.load_state_dict(best_state)
    model.eval()

    plot_loss_evolution(train_losses, len(train_losses),
                        title=f'{model_name}: Evolución del Loss (MAE) — pagerank_norm',
                        val_losses=val_losses)

    with torch.no_grad():
        pred_final = model(features, gnn.graph.edge_index)
        embeddings = model.encode(features, gnn.graph.edge_index)

    pred_test   = pred_final[test_mask].cpu().numpy().flatten()
    labels_test = labels[test_mask].cpu().numpy().flatten()
    test_mae    = loss_fn(pred_final[test_mask], labels[test_mask]).item()
    r2          = r2_score(labels_test, pred_test)

    plt.figure(figsize=(5, 4))
    plt.scatter(labels_test, pred_test, alpha=0.15, s=4)
    plt.xlabel('Real (pagerank norm)')
    plt.ylabel('Predicho')
    plt.title(f'{model_name}: Real vs Predicho')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'\n  [Métricas {model_name}]')
    print(f'  Best Val MAE : {best_val_loss:.6f}')
    print(f'  Test MAE     : {test_mae:.6f}')
    print(f'  Test R²      : {r2:.6f}')

    torch.save(model.state_dict(),
               f'{save_dir}/model_emb_{model_name}_{attr}.pth')
    torch.save(embeddings,
               f'{save_dir}/embeddings_ribs_{model_name}_{attr}.pt')
    print(f'  Guardado: embeddings_ribs_{model_name}_{attr}.pt')
    print()

In [ ]:
# # Guardar los mismos embeddings con el prefijo 'mlp' para mantener
# # la convención de nombres usada en Caso 0 / Caso 1.
# # El entrenamiento es node-regression en ambas variantes; el sufijo
# # indica la nomenclatura del decoder simbólico del enfoque staged.

# save_dir = data_path + 'results/EnfoquePorPartes/Caso4'
# attr     = 'pagerank_norm'

# for model_name in ['GCN', 'GraphSAGE', 'GAT']:
#     src_path = f'{save_dir}/embeddings_ribs_{model_name}_{attr}.pt'
#     dst_path = f'{save_dir}/embeddings_mlp_{model_name}_{attr}.pt'
#     if os.path.exists(src_path):
#         import shutil
#         shutil.copy2(src_path, dst_path)
#         print(f'  Copiado: embeddings_mlp_{model_name}_{attr}.pt')
#     else:
#         print(f'  [AVISO] No encontrado: {src_path}  — ejecuta la celda anterior primero.')


In [ ]:


# Distribución de PageRank normalizado (min-max)
pr_norm = gnn.graph.pagerank_norm.squeeze().detach().cpu().numpy()

plt.figure(figsize=(8, 4))
plt.hist(pr_norm, bins=100, alpha=0.85, edgecolor='black', linewidth=0.3)
plt.title('Distribución de PageRank normalizado')
plt.xlabel('PageRank normalizado')
plt.ylabel('Frecuencia')
plt.grid(True, linestyle='--', alpha=0.35)
plt.tight_layout()
plt.show()



### Caso 5: Creación de embeddings a partir de DeepWalk y BGP2Vec 

### Caso 5.1: DeepWalk

In [ ]:
gnn = GNN(debug=True)
gnn.load_dataset(str(nodes_file), str(edges_file))
NUM_EPOCHS = 50


In [ ]:
from torch_geometric.nn import Node2Vec
from torch.optim import SparseAdam

# Configuración del modelo DeepWalk (Node2Vec con p=q=1 es DeepWalk)
# -------------------------------------
# context_size=2 equivale a window_size=1 (ventana hacia adelante);
# num_negative_samples=5 equivale al negative_size por defecto de DGL.
model = Node2Vec(
    gnn.graph.edge_index,
    embedding_dim=32,
    walk_length=40,
    context_size=2,
    walks_per_node=1,
    p=1.0, q=1.0,
    num_negative_samples=5,
    sparse=True
)

# DataLoader para muestreo de caminatas aleatorias
# -------------------------------------------------
dataloader = model.loader(batch_size=128, shuffle=True)

optimizer = SparseAdam(list(model.parameters()), lr=0.01)

# Entrenamiento
# -------------------------------------
losses = []

print("=" * 60)
print("Iniciando entrenamiento de DeepWalk...")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    num_batches = 0
    
    for pos_rw, neg_rw in dataloader:
        # Forward + backward (caminatas positivas y negativas ya muestreadas por PyG)
        loss = model.loss(pos_rw, neg_rw)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    # Calcular pérdida promedio de la época
    avg_loss = epoch_loss / num_batches if num_batches > 0 else 0
    losses.append(avg_loss)
    
    # Mostrar progreso
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS}: Loss = {avg_loss:.6f}")

print("\n" + "=" * 60)
print("Entrenamiento completado!")
print("=" * 60)

# Obtener embeddings finales
# -------------------------------------
embeddings = model.embedding.weight.detach().cpu()

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Número de nodos: {gnn.graph.num_nodes}")
print(f"Dimensión de embeddings: {embeddings.shape[1]}")
print(f"\nLoss inicial: {losses[0]:.6f}")
print(f"Loss final: {losses[-1]:.6f}")
print(f"Reducción: {((losses[0] - losses[-1]) / losses[0] * 100):.2f}%")

# 38 min
# 35 min

Evolución del Loss durante el entrenamiento

In [ ]:
plot_loss_evolution(losses, NUM_EPOCHS, title='DeepWalk: Evolución del Loss durante Entrenamiento')

In [ ]:
MES = "marzo"
ANO = "2026"

In [ ]:
# Crear directorio si no existe
import os
save_dir = data_path + 'results/EnfoquePorPartes/Caso5/'
os.makedirs(save_dir, exist_ok=True)

# Guardar los embeddings
embeddings_path = save_dir + f'embeddings_deepWalk_{MES}_{ANO}.pt'
torch.save(embeddings, embeddings_path)

# Guardar el modelo completo
model_path = save_dir + f'model_deepWalk_{MES}_{ANO}.pth'
torch.save(model.state_dict(), model_path)

print('=' * 60)
print('ARCHIVOS GUARDADOS')
print('=' * 60)
print(f'Embeddings guardados en: {embeddings_path}')
print(f'Modelo guardado en: {model_path}')
print(f'\nShape embeddings: {embeddings.shape}')


### Caso 5.2 BGP2Vec

In [ ]:
from gensim.models import Word2Vec


In [ ]:
MODELS_PATH = data_path + 'results/EnfoquePorPartes/Caso5/'
print(f"[Modelos guardados en]: {MODELS_PATH}]")
print(f"[Archivo con RIBs]: {ribs_file}")



#### Importar rutas BGP 

In [ ]:
from modules.bgp2vec import RIBCorpus
import itertools

# Configuración
MAX_ROUTES_TRAIN = 5_000_000   # None => usar todas las rutas
COUNT_ALL_ROUTES = True       # True => recorre todo el archivo para contar

# Muestra pequeña para verificar formato
ribs_file =  data_path + "rib_20260301_0000_to_20260301_1200.txt" 
sample = list(itertools.islice(RIBCorpus(ribs_file, skip_header=True), 10))
print("[RUTAS BGP - MUESTRA]:", sample)

# Conteo total opcional (puede tardar porque recorre todo)
if COUNT_ALL_ROUTES:
    count_routes = sum(1 for _ in RIBCorpus(ribs_file, skip_header=True))
    print("[CANTIDAD TOTAL DE RUTAS BGP]:", count_routes)
else:
    print("[CANTIDAD TOTAL DE RUTAS BGP]: omitido (COUNT_ALL_ROUTES=False)")

# Corpus para entrenamiento: limitado o completo
if MAX_ROUTES_TRAIN is None:
    bgp_routes = RIBCorpus(ribs_file, skip_header=True)
    print("[RUTAS PARA ENTRENAR]: todas")
else:
    bgp_routes = itertools.islice(RIBCorpus(ribs_file, skip_header=True), MAX_ROUTES_TRAIN)
    print(f"[RUTAS PARA ENTRENAR]: primeras {MAX_ROUTES_TRAIN}")

print("[TIPO]:", type(bgp_routes))

#### Crear Embeddings

In [ ]:
MODELS_PATH = data_path + 'results/EnfoquePorPartes/Caso5/'

In [ ]:
data_path

In [ ]:
from modules.bgp2vec import BGP2VEC

test_limit = 27000000 #cantidad paths/horaciones
mode = None# 'test' # par limitar cantidad de paths/oraciones
epochs = 1
debug = True

# Crea embeddings de ASN con  BGP2VEC y lo guarda en "bgp2vec/bgp2vec.word2vec"
bgp2vec = BGP2VEC(model_path = MODELS_PATH + 'bgp2vec.word2vec',
                  bgp_routes=bgp_routes,
                  rewrite=True, 
                  test_limit= test_limit, 
                  mode = mode, 
                  epochs = epochs)

print("[MODELO]",bgp2vec.model)
print("[NUMERO ASN REPETIDOS Y TODO]",bgp2vec.model.corpus_total_words)

# 1.000.000 -> 7 seg
# 2.000.000 -> 14 seeg
# todo el dataset (27.000.000) -> 5 min

In [ ]:
# Importar embeddings preentrenado de los ASN  con BGP2VEC
BGP2Vec_model = Word2Vec.load(MODELS_PATH + "bgp2vec.word2vec")

# Obtener la matriz de embeddings
embeddings = BGP2Vec_model.wv.vectors
total_ASNs, embedding_vecor_length = embeddings.shape


print(f"[TOTAL ASN]: {total_ASNs}" )
print(f"[EMBEDDINGS LEN]: {embedding_vecor_length}")
print(f"[EMBEDDINGS] {embeddings}")

In [ ]:
# Mapeo ASN -> ID usado en los embeddings
asn_to_index = BGP2Vec_model.wv.key_to_index
print(f"[ASN TO INDEX]: {asn_to_index}")
print(f"[TAMANO]: {len(asn_to_index)}")

## Clasificación Aristas 

In [ ]:
# !pip uninstall gensim -y 
# %pip install gensim==4.3.3

In [ ]:
# !pip install scikit-learn
# !pip install numpy
# !pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
# # !pip uninstall tensorflow -y
# !pip install tensorflow==2.17.0
# !pip install matplotlib
# !pip install  dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html



In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
from collections import Counter
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
import torch



In [ ]:
TOR_LABELS_DICT = {'P2P':0, 'C2P': 1,'P2C': 2}
TOR_CSV_LABELS_DICT = {'P2P':0,'P2C': -1}

TESIS_PATH  = '/media/vale/KINGSTON/TESIS/'
class_names = ['P2P', 'C2P', 'P2C']

# Dataset entrenamiento RNA para relaciones entre AS
MODEL_NAME = 'NN_Inference'
MES = 'marzo'



## Importar Emberddings

In [ ]:
# Seleccionar embeddings a cargar
# Descomenta el bloque del caso que quieras usar

# # # ─── Caso 0: link prediction, atributos PeeringDB (mis_attr_marzo) ───────────
attr = 'mis_attr_marzo'
CASO = "Caso0"
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_DotProduct_GCN_mis_attr_marzo.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_DotProduct_GraphSAGE_mis_attr_marzo.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_DotProduct_GAT_mis_attr_marzo.pt'
embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_MLP_GCN_mis_attr_marzo.pt'
embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_MLP_GraphSAGE_mis_attr_marzo.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_MLP_GAT_mis_attr_marzo.pt'

# ─── Caso 1: link prediction, atributos grado entrada/salida ──────────────────
# attr = 'grado_attr_febrero'
# CASO = 'Caso1'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_DotProduct_GCN_grado_attr_febrero.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_DotProduct_GraphSAGE_grado_attr_febrero.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_DotProduct_GAT_grado_attr_febrero.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_MLP_GCN_grado_attr_febrero.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_MLP_GraphSAGE_grado_attr_febrero.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_MLP_GAT_grado_attr_febrero.pt'

# ─── Caso 3: regresión grado de salida ───────────────────────────────────────
# attr = 'out_degree'
# CASO = 'Caso3'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_GCN_out_degree.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_GraphSAGE_out_degree.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_GAT_out_degree.pt'

# ─── Caso 4: regresión PageRank ───────────────────────────────────────────────
# attr = 'pagerank_norm'
# CASO = 'Caso4'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_GCN_pagerank_norm.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_GraphSAGE_pagerank_norm.pt'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_ribs_GAT_pagerank_norm.pt'

# ─── Caso 5.1: DeepWalk ─────────────────────────────────────────────────────────
# attr = 'deepWalk'
# CASO = 'Caso5'
# embeddings_path = data_path +  f'results/EnfoquePorPartes/{CASO}/embeddings_deepWalk_marzo_2026.pt'
# embeddings_path =  "/home/vale/Escritorio/GIT/TrabajoTesis/results/EnfoquePorPartes/Caso5/embeddings_deepWalk_marzo_2026.pt"
# # Modelo guardado en: /home/vale/Escritorio/GIT/TrabajoTesis/results/EnfoquePorPartes/Caso5/model_deepWalk_marzo_2026.pth
# print('Cargando:', embeddings_path)

# ─── Caso 5.2: BGP2Vec ───────────────────────────────────────────────

# Caso 5: BGP2Vvec
attr = 'bgp2vec'
CASO = "Caso5"

save_dir = data_path + 'results/EnfoquePorPartes/Caso5/'
embeddings_path = data_path + 'results/EnfoquePorPartes/Caso5/bgp2vec.word2vec'



In [ ]:
save_dir = data_path + f'results/EnfoquePorPartes/{CASO}/'

if attr != 'bgp2vec':
    print ("=" * 60)
    print(f"Cargando embeddings de: {embeddings_path}")
    print ("=" * 60)
    embeddings_name = embeddings_path.split('/')[-1].split('.')[0]  # Nombre del archivo sin extensión
    embeddings_pt = torch.load(embeddings_path)  # asegúrate que sea un tensor
    embeddings = embeddings_pt.detach().numpy()

if attr == 'bgp2vec':
    from gensim.models import Word2Vec
    # Si son los mbddings de BGP2Vec, cargamos con gensim
    print ("=" * 60)
    print(f"Cargando embeddings BGP2Vec: {embeddings_path}")
    print ("=" * 60)
    embeddings_name = "bgp2vec"  # Nombre del archivo sin extensión

    BGP2Vec_model = Word2Vec.load(embeddings_path)

    # Obtener la matriz de embeddings
    embeddings = BGP2Vec_model.wv.vectors

    asn_to_index = BGP2Vec_model.wv.key_to_index
    print(f"[ASN TO INDEX]: {asn_to_index}")
    print(f"[TAMANO]: {len(asn_to_index)}")

total_ASNs, embedding_vecor_length = embeddings.shape


print(f"[TOTAL ASN]: {total_ASNs}" )
print(f"[EMBEDDINGS LEN]: {embedding_vecor_length}")
print(f"[EMBEDDINGS] {embeddings}")





## Clasificación de Tipos de Enlace con ANN

Se cargan los embeddings GNN ya entrenados y el dataset ToR (pares de ASNs con etiqueta P2P/C2P/P2C).
Para cada par (ASN_u, ASN_v) se construye un vector de arista **[e_u | e_v | e_u−e_v | e_u·e_v]** que se pasa a un clasificador MLP.

El notebook está organizado en:
1. Carga y exploración del dataset ToR
2. Mapeo ASN → node_id y construcción vectorizada de features de arista
3. Definición del modelo ANN
4. Entrenamiento **sin batches** (dataset completo en memoria)
5. Entrenamiento **con batches** (DataLoader)
6. Evaluación final: accuracy, reporte por clase, matrices de confusión

### 1. Configuración

In [ ]:
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
MES  = 'marzo'
ANO  = '2026'
TESIS_PATH = '/media/vale/KINGSTON/TESIS/'
TOR_PATH   = TESIS_PATH + f'data_2026/ToR_data/tor_data_{MES}/'

CLASS_NAMES = ['P2P', 'C2P', 'P2C']
N_CLASSES   = 3

# Hiperparámetros ANN
ANN_HIDDEN   = 256
ANN_DROPOUT  = 0.3
ANN_LR       = 1e-3 # 0,001
ANN_EPOCHS   = 200
ANN_PATIENCE = 50
ANN_BATCH    = 512 #4096
VAL_SIZE     = 0.2



### 2. Cargar Dataset ToR

In [ ]:
# x contiene pares de ASNs (strings), y contiene etiquetas 0/1/2
x_train_raw = np.load(TOR_PATH + 'x_training.npy')          # (N, 2) dtype=str
x_test_raw  = np.load(TOR_PATH + 'x_test.npy')
y_train_raw = np.load(TOR_PATH + 'y_training.npy').astype(int)
y_test_raw  = np.load(TOR_PATH + 'y_test.npy').astype(int)

print(f'Train: {x_train_raw.shape}  dtype={x_train_raw.dtype}')
print(f'Test:  {x_test_raw.shape}')
print(f'Distribución y_train: {dict(sorted(Counter(y_train_raw.tolist()).items()))}')
print(f'Distribución y_test:  {dict(sorted(Counter(y_test_raw.tolist()).items()))}')
print(f'Ejemplo x_train[0]: ASN_u={x_train_raw[0,0]}  ASN_v={x_train_raw[0,1]}')


### 3. Mapeo ASN → node_id y Features de Arista

Los embeddings están indexados por `node_id` (fila en el grafo DGL), pero el dataset ToR usa ASNs.
Se carga el CSV de nodos para construir el diccionario `asn → node_id`.

In [ ]:
if attr != 'bgp2vec':
    print ("=" * 60)
    print(f"Mapeo de nodos en el grafo a partir del CSV de nodos: {nodes_file}")
    print ("=" * 60)

    # Construir mapeo {asn(int) -> node_id(int)} desde el CSV de nodos
    nodes_df = pd.read_csv(str(nodes_file))
    asn_to_nodeid = dict(zip(nodes_df['asn'].astype(int), nodes_df['node_id'].astype(int)))

if attr == 'bgp2vec':
    print ("=" * 60)
    print(f"Mapeo de nodos en el grafo a partir de bgp2vec: {nodes_file}")
    print ("=" * 60)

    raw_index = BGP2Vec_model.wv.key_to_index
    asn_to_nodeid = {}
    invalid_tokens = []

    for k, v in raw_index.items():
        k_str = str(k).strip()
        if not k_str:
            invalid_tokens.append(k)
            continue
        try:
            asn_to_nodeid[int(k_str)] = v
        except ValueError:
            invalid_tokens.append(k)

    print(f"[ASN válidos]: {len(asn_to_nodeid):,}")
    print(f"[ASN inválidos descartados]: {len(invalid_tokens):,}")
    if invalid_tokens:
        print(f"[Muestra inválidos]: {invalid_tokens[:10]}")

print(f'[Nodos en el grafo]:    {len(asn_to_nodeid):,}')
print(f'[ASNs en x_train]:      {len(set(x_train_raw.flatten())):,} únicos')

# Cobertura: ¿cuántos ASNs del ToR están en el grafo?
asns_tor = set(int(a) for a in x_train_raw.flatten())
covered = asns_tor & set(asn_to_nodeid.keys())
print(f'[Cobertura]:            {len(covered):,} / {len(asns_tor):,} ({len(covered)/len(asns_tor)*100:.1f}%)')

In [ ]:
def build_edge_features(x_pairs, emb_tensor, asn_to_nodeid):
    """
    Construye features de arista [e_u | e_v | e_u-e_v | e_u*e_v] de forma vectorizada.

    Args:
        x_pairs       : (N, 2) array de ASNs (strings o ints)
        emb_tensor    : (num_nodes, emb_dim) float tensor o numpy array
        asn_to_nodeid : dict {asn(int|str) -> node_id(int)}

    Returns:
        feats  : (M, 4*emb_dim) float tensor  (M <= N, solo pares válidos)
        valid  : (N,) bool array  (True si ambos ASNs están en el grafo)
    """
    # Asegurar tensor PyTorch para operaciones posteriores (torch.cat)
    emb_tensor = torch.as_tensor(emb_tensor, dtype=torch.float32)

    def _lookup_nid(asn_val):
        asn_int = int(asn_val)
        # Compatibilidad: diccionarios con claves int (GNN) o str (BGP2Vec)
        return asn_to_nodeid.get(asn_int, asn_to_nodeid.get(str(asn_int), -1))

    nid_u = np.array([_lookup_nid(a) for a in x_pairs[:, 0]], dtype=np.int64)
    nid_v = np.array([_lookup_nid(a) for a in x_pairs[:, 1]], dtype=np.int64)
    valid = (nid_u >= 0) & (nid_v >= 0)

    # Si no hay pares válidos, devolver tensor vacío con dimensión correcta
    if valid.sum() == 0:
        empty_feats = torch.empty((0, emb_tensor.shape[1] * 4), dtype=emb_tensor.dtype)
        return empty_feats, valid

    idx_u = torch.as_tensor(nid_u[valid], dtype=torch.long)
    idx_v = torch.as_tensor(nid_v[valid], dtype=torch.long)

    e_u = emb_tensor[idx_u]          # (M, D)
    e_v = emb_tensor[idx_v]          # (M, D)
    feats = torch.cat([e_u, e_v, e_u - e_v, e_u * e_v], dim=1)  # (M, 4D)
    return feats, valid


# embeddings cargado en la sección 'Importar Embeddings' (celdas anteriores)
emb_tensor = torch.as_tensor(embeddings, dtype=torch.float32)
emb_dim    = emb_tensor.shape[1]
print(f'Embeddings: {tuple(emb_tensor.shape)}  →  features por arista: {4 * emb_dim}')

print('Construyendo features de entrenamiento...')
X_train_feat, train_valid = build_edge_features(x_train_raw, emb_tensor, asn_to_nodeid)
y_train_feat = y_train_raw[train_valid]
print(f'  {x_train_raw.shape[0]:,} pares → {train_valid.sum():,} válidos '
      f'({train_valid.mean()*100:.1f}%)')

print('Construyendo features de test...')
X_test_feat, test_valid = build_edge_features(x_test_raw, emb_tensor, asn_to_nodeid)
y_test_feat = y_test_raw[test_valid]
print(f'  {x_test_raw.shape[0]:,} pares → {test_valid.sum():,} válidos '
      f'({test_valid.mean()*100:.1f}%)')

### 4. Preparar Tensores y Pesos de Clase

In [ ]:
# Train / Val split estratificado (80/20 del training)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_feat, y_train_feat,
    test_size=VAL_SIZE, random_state=42, stratify=y_train_feat)

y_tr_t  = torch.tensor(y_tr,  dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_te_t  = torch.tensor(y_test_feat, dtype=torch.long)

print(f'Train: {len(X_tr):,}  Val: {len(X_val):,}  Test: {len(X_test_feat):,}')
print(f'y_tr  dist: {dict(sorted(Counter(y_tr.tolist()).items()))}')

# Pesos de clase para compensar desbalance P2P >> C2P, P2C
cw = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_tr)
class_weights = torch.tensor(cw, dtype=torch.float)
print(f'Class weights: P2P={cw[0]:.3f}  C2P={cw[1]:.3f}  P2C={cw[2]:.3f}')


### 5.1 Definición del Modelo ANN

El vector de entrada es **[e_u | e_v | e_u−e_v | e_u·e_v]** (4×emb_dim).
La diferencia y el producto capturan la **asimetría direccional** necesaria para separar C2P de P2C.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class EdgeClassifierANN(nn.Module):
    """
    MLP que clasifica el tipo de enlace AS (P2P / C2P / P2C)
    a partir del vector de arista [e_u | e_v | e_u-e_v | e_u*e_v].
    """
    def __init__(self, emb_dim, n_classes=3, hidden=256, dropout=0.3):
        super().__init__()
        in_dim = 4 * emb_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, x):    # x: (batch, 4*emb_dim)
        return self.net(x)


# Verificar dimensiones
_m = EdgeClassifierANN(emb_dim, N_CLASSES, ANN_HIDDEN, ANN_DROPOUT)
_x = torch.randn(4, 4 * emb_dim)
print(f'Test forward: input={_x.shape}  output={_m(_x).shape}')
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {n_params:,}')
del _m, _x


### 5.2 Definición y entrenamieneto del Modelo CNN 

In [ ]:
# import torch
# import torch.nn as nn
# from torch.utils.data import TensorDataset, DataLoader


# class EdgeClassifierCNN(nn.Module):
#     """
#     CNN 1D que clasifica el tipo de enlace AS (P2P / C2P / P2C) a partir del
#     vector de arista pre-calculado [e_u | e_v | e_u-e_v | e_u*e_v].

#     Trata los 4 componentes como una secuencia de 4 'tokens' de dimensión emb_dim
#     y aplica Conv1D para capturar patrones locales entre ellos.

#     Input:  (batch, 4 * emb_dim)  — mismo formato que EdgeClassifierANN
#     """

#     def __init__(self, emb_dim, n_classes=3, dropout=0.3):
#         super().__init__()
#         self.emb_dim = emb_dim

#         # Reshape interno: (batch, 4*emb_dim) → (batch, emb_dim, 4)
#         self.conv1 = nn.Conv1d(emb_dim, 64, kernel_size=2, padding=1)
#         self.pool1 = nn.MaxPool1d(kernel_size=2)

#         self.conv2 = nn.Conv1d(64, 32, kernel_size=2, padding=0)
#         self.pool2 = nn.AdaptiveMaxPool1d(1)   # → (batch, 32, 1)

#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(32, 100),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(100, n_classes),
#         )

#     def forward(self, x):                       # x: (batch, 4*emb_dim)
#         b = x.size(0)
#         x = x.view(b, 4, self.emb_dim)         # (batch, 4, emb_dim)
#         x = x.permute(0, 2, 1)                 # (batch, emb_dim, 4)
#         x = torch.relu(self.conv1(x))           # (batch, 64, ≤5)
#         x = self.pool1(x)                       # (batch, 64, ≤2)
#         x = torch.relu(self.conv2(x))           # (batch, 32, ≤1)
#         x = self.pool2(x)                       # (batch, 32, 1)
#         return self.classifier(x)


In [ ]:
# def train_cnn(model, X_tr, y_tr_t, X_val, y_val_t, X_test, y_te_t,
#               class_weights=None,
#               CNN_BATCH=64, CNN_EPOCHS=100, CNN_LR=1e-3,
#               CNN_PATIENCE=10, N_CLASSES=3, CLASS_NAMES=None):
#     """
#     Entrenamiento CNN con early stopping y evaluación final en test.
#     Acepta features de arista pre-calculadas (batch, 4*emb_dim).
#     """
#     from sklearn.metrics import accuracy_score, classification_report

#     loss_fn = nn.CrossEntropyLoss(weight=class_weights)

#     ds_train   = TensorDataset(X_tr,  y_tr_t)
#     ds_val     = TensorDataset(X_val, y_val_t)
#     loader_tr  = DataLoader(ds_train, batch_size=CNN_BATCH,     shuffle=True,  drop_last=False)
#     loader_val = DataLoader(ds_val,   batch_size=CNN_BATCH * 4, shuffle=False)

#     opt   = torch.optim.Adam(model.parameters(), lr=CNN_LR)
#     sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
#         opt, mode='min', factor=0.5, patience=5, verbose=False)

#     train_losses, val_losses = [], []
#     best_val, best_state, patience_ctr = float('inf'), None, 0

#     print('=' * 60)
#     print(f'CNN  (batch={CNN_BATCH})')
#     print('=' * 60)

#     for epoch in range(CNN_EPOCHS):

#         # ── Train ──────────────────────────────────────────────────────
#         model.train()
#         epoch_loss = 0
#         for x_b, y_b in loader_tr:
#             logits = model(x_b)
#             loss   = loss_fn(logits, y_b)
#             opt.zero_grad(); loss.backward(); opt.step()
#             epoch_loss += loss.item()
#         avg_train = epoch_loss / len(loader_tr)

#         # ── Validation ─────────────────────────────────────────────────
#         model.eval()
#         val_loss, n_correct = 0, 0
#         with torch.no_grad():
#             for x_b, y_b in loader_val:
#                 logits    = model(x_b)
#                 val_loss += loss_fn(logits, y_b).item()
#                 n_correct += (logits.argmax(1) == y_b).sum().item()
#         avg_val = val_loss / len(loader_val)
#         val_acc = n_correct / len(X_val)

#         train_losses.append(avg_train)
#         val_losses.append(avg_val)
#         sched.step(avg_val)

#         if epoch % 10 == 0 or epoch == CNN_EPOCHS - 1:
#             print(f'  Epoch {epoch:3d}: train={avg_train:.4f}  '
#                   f'val={avg_val:.4f}  val_acc={val_acc*100:.2f}%')

#         # ── Early stopping ─────────────────────────────────────────────
#         if avg_val < best_val:
#             best_val    = avg_val
#             best_state  = {k: v.clone() for k, v in model.state_dict().items()}
#             patience_ctr = 0
#         else:
#             patience_ctr += 1
#         if patience_ctr >= CNN_PATIENCE:
#             print(f'  Early stopping en epoch {epoch}.')
#             break

#     model.load_state_dict(best_state)
#     print(f'Mejor val loss: {best_val:.4f}')

#     # ── Evaluación en test ─────────────────────────────────────────────
#     print('=' * 60)
#     print('EVALUACIÓN EN TEST SET')
#     print('=' * 60)

#     model.eval()
#     all_preds = []
#     loader_te = DataLoader(TensorDataset(X_test), batch_size=CNN_BATCH * 4)
#     with torch.no_grad():
#         for (x_b,) in loader_te:
#             all_preds.append(model(x_b).argmax(1))
#     preds_test = torch.cat(all_preds).numpy()
#     y_te_np    = y_te_t.numpy()

#     print(f'Accuracy: {accuracy_score(y_te_np, preds_test)*100:.2f}%\n')
#     if CLASS_NAMES:
#         print(classification_report(y_te_np, preds_test, target_names=CLASS_NAMES))

#     return model, train_losses, val_losses


In [ ]:
# CNN_BATCH    = 4096
# CNN_EPOCHS   = 100
# CNN_LR       = 1e-3
# CNN_PATIENCE = 20
# CNN_DROPOUT  = 0.3

# model_cnn = EdgeClassifierCNN(emb_dim, n_classes=N_CLASSES, dropout=CNN_DROPOUT)

# model_cnn, tr_losses_cnn, val_losses_cnn = train_cnn(
#     model_cnn,
#     X_tr, y_tr_t,
#     X_val, y_val_t,
#     X_test_feat, y_te_t,
#     class_weights=class_weights,
#     CNN_BATCH=CNN_BATCH, CNN_EPOCHS=CNN_EPOCHS, CNN_LR=CNN_LR,
#     CNN_PATIENCE=CNN_PATIENCE, N_CLASSES=N_CLASSES, CLASS_NAMES=CLASS_NAMES,
# )


#### Curvas de Loss — CNN

In [ ]:
# fig, ax = plt.subplots(figsize=(8, 4))
# ax.plot(tr_losses_cnn,  label='Train', linewidth=1.5)
# ax.plot(val_losses_cnn, label='Val',   linewidth=1.5)
# ax.set_title(f'Evolución del Loss – CNN Clasificador AS ({embeddings_name})')
# ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
# ax.legend(); ax.grid(alpha=0.3)
# plt.tight_layout()
# plt.savefig(save_dir + f'loss_cnn_{embeddings_name}.png', dpi=150)
# plt.show()


### 6. Entrenamiento Sin Batches

Todo el conjunto de entrenamiento se pasa de una vez en cada época. Funciona bien si los datos caben en GPU/RAM.

In [ ]:
# # Entrenamiento full-batch (sin DataLoader)
# loss_fn_nb = nn.CrossEntropyLoss(weight=class_weights)
# model_nb = EdgeClassifierANN(emb_dim, N_CLASSES, ANN_HIDDEN, ANN_DROPOUT)
# opt_nb = torch.optim.Adam(model_nb.parameters(), lr=ANN_LR)
# sched_nb = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     opt_nb, mode='min', factor=0.5, patience=5, verbose=False)

# train_losses_nb, val_losses_nb = [], []
# best_val_nb, best_state_nb, patience_ctr_nb = float('inf'), None, 0

# print('=' * 60)
# print('Sin Batches (full-batch)')
# print('=' * 60)
# for epoch in range(ANN_EPOCHS):
#     # ── Train ──
#     model_nb.train()
#     logits_tr = model_nb(X_tr)
#     loss_tr = loss_fn_nb(logits_tr, y_tr_t)
#     opt_nb.zero_grad()
#     loss_tr.backward()
#     opt_nb.step()
#     avg_train = loss_tr.item()

#     # ── Val ──
#     model_nb.eval()
#     with torch.no_grad():
#         logits_val = model_nb(X_val)
#         avg_val = loss_fn_nb(logits_val, y_val_t).item()
#         val_acc = (logits_val.argmax(1) == y_val_t).float().mean().item()

#     train_losses_nb.append(avg_train)
#     val_losses_nb.append(avg_val)
#     sched_nb.step(avg_val)

#     if epoch % 10 == 0 or epoch == ANN_EPOCHS - 1:
#         print(f'Epoch {epoch:3d}: train={avg_train:.4f}  val={avg_val:.4f}  val_acc={val_acc*100:.2f}%')

#     # ── Early stopping ──
#     if avg_val < best_val_nb:
#         best_val_nb = avg_val
#         best_state_nb = {k: v.clone() for k, v in model_nb.state_dict().items()}
#         patience_ctr_nb = 0
#     else:
#         patience_ctr_nb += 1

#     if patience_ctr_nb >= ANN_PATIENCE:
#         print(f'Early stopping epoch {epoch}.')
#         break

# if best_state_nb is not None:
#     model_nb.load_state_dict(best_state_nb)

# print(f'Mejor val loss (sin batches): {best_val_nb:.4f}')
# print(f"Mejor accuracy en val (sin batches): {(logits_val.argmax(1) == y_val_t).float().mean().item()*100:.2f}%")

# #################################################################################
# # Elegir modelo: sin batches 
# #################################################################################
# print("=" * 60)
# print('Evaluación en test set')
# print("=" * 60)

# best_model = model_nb  #model_nb

# best_model.eval()
# all_preds = []
# loader_te = DataLoader(TensorDataset(X_test_feat), batch_size=ANN_BATCH * 4)
# with torch.no_grad():
#     for (x_b,) in loader_te:
#         all_preds.append(best_model(x_b).argmax(1))
# preds_test = torch.cat(all_preds).numpy()
# y_te_np    = y_te_t.numpy()

# print('=' * 60)
# print('EVALUACIÓN EN TEST SET')
# print('=' * 60)
# print(f'Accuracy: {accuracy_score(y_te_np, preds_test)*100:.2f}%\n')
# print(classification_report(y_te_np, preds_test, target_names=CLASS_NAMES))

### 7. Entrenamiento Con Batches (DataLoader)

Más eficiente en memoria y con mejor generalización gracias al ruido de mini-batch.

In [ ]:
loss_fn_b = nn.CrossEntropyLoss(weight=class_weights)
# loss_fn_b = nn.CrossEntropyLoss()


ds_train   = TensorDataset(X_tr, y_tr_t)
ds_val     = TensorDataset(X_val, y_val_t)
loader_tr  = DataLoader(ds_train, batch_size=ANN_BATCH, shuffle=True,  drop_last=False)
loader_val = DataLoader(ds_val,   batch_size=ANN_BATCH * 4, shuffle=False)

model_b  = EdgeClassifierANN(emb_dim, N_CLASSES, ANN_HIDDEN, ANN_DROPOUT)
opt_b    = torch.optim.Adam(model_b.parameters(), lr=ANN_LR)
sched_b  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_b, mode='min', factor=0.5, patience=5, verbose=False)

train_losses_b, val_losses_b = [], []
train_accs_b,  val_accs_b  = [], []
best_val_b, best_state_b, patience_ctr_b = float('inf'), None, 0

print("=" * 60)
print(f'Con Batches (batch={ANN_BATCH}) ')
print("=" * 60)
for epoch in range(ANN_EPOCHS):
    # ── Train ──
    model_b.train()
    epoch_loss = 0
    for x_b, y_b in loader_tr:
        logits = model_b(x_b)
        loss   = loss_fn_b(logits, y_b)
        opt_b.zero_grad(); loss.backward(); opt_b.step()
        epoch_loss += loss.item()
    avg_train = epoch_loss / len(loader_tr)
    with torch.no_grad():
        tr_correct = sum(
            (model_b(x_b).argmax(1) == y_b).sum().item()
            for x_b, y_b in loader_tr
        )
    train_acc = tr_correct / len(X_tr)


    # ── Val ──
    model_b.eval()
    val_loss, n_correct = 0, 0
    with torch.no_grad():
        for x_b, y_b in loader_val:
            logits = model_b(x_b)
            val_loss  += loss_fn_b(logits, y_b).item()
            n_correct += (logits.argmax(1) == y_b).sum().item()
    avg_val  = val_loss / len(loader_val)
    val_acc  = n_correct / len(X_val)
    train_losses_b.append(avg_train)
    val_losses_b.append(avg_val)
    train_accs_b.append(train_acc)
    val_accs_b.append(val_acc)
    sched_b.step(avg_val)

    if epoch % 10 == 0 or epoch == ANN_EPOCHS - 1:
        print(f'  Epoch {epoch:3d}: train={avg_train:.4f}  '
              f'val={avg_val:.4f}  val_acc={val_acc*100:.2f}%')

    # ── Early stopping ──
    if avg_val < best_val_b:
        best_val_b   = avg_val
        best_state_b = {k: v.clone() for k, v in model_b.state_dict().items()}
        patience_ctr_b = 0
    else:
        patience_ctr_b += 1
    if patience_ctr_b >= ANN_PATIENCE:
        print(f'  Early stopping epoch {epoch}.')
        break

model_b.load_state_dict(best_state_b)
print(f'Mejor val loss (con batches): {best_val_b:.4f}')

# ── Gráfico: Loss y Accuracy durante entrenamiento ──────────────
epochs_ran = len(train_losses_b)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(range(1, epochs_ran + 1), train_losses_b, label="Train Loss")
ax1.plot(range(1, epochs_ran + 1), val_losses_b,   label="Val Loss")
ax1.set_xlabel("Época")
ax1.set_ylabel("Loss")
ax1.set_title("ANN — Evolución del Loss")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(range(1, epochs_ran + 1), [a * 100 for a in train_accs_b], label="Train Acc")
ax2.plot(range(1, epochs_ran + 1), [a * 100 for a in val_accs_b],   label="Val Acc")
ax2.set_xlabel("Época")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("ANN — Evolución de la Accuracy")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


############################################################
# Elegir modelo: con batches suele generalizar mejor
############################################################

print("=" * 60)
print('Evaluación en test set')
print("=" * 60)

best_model = model_b  #model_nb

best_model.eval()
all_preds = []
loader_te = DataLoader(TensorDataset(X_test_feat), batch_size=ANN_BATCH * 4)
with torch.no_grad():
    for (x_b,) in loader_te:
        all_preds.append(best_model(x_b).argmax(1))
preds_test = torch.cat(all_preds).numpy()
y_te_np    = y_te_t.numpy()

print('=' * 60)
print('EVALUACIÓN EN TEST SET')
print('=' * 60)
print(f'Accuracy: {accuracy_score(y_te_np, preds_test)*100:.2f}%\n')
print(classification_report(y_te_np, preds_test, target_names=CLASS_NAMES))

### 8. Evaluación Final

Se evalúa el modelo con batches (mejor opción) sobre el conjunto de test.

In [ ]:
# Elegir modelo: con batches suele generalizar mejor
best_model = model_b  #model_nb

best_model.eval()
all_preds = []
loader_te = DataLoader(TensorDataset(X_test_feat), batch_size=ANN_BATCH * 4)
with torch.no_grad():
    for (x_b,) in loader_te:
        all_preds.append(best_model(x_b).argmax(1))
preds_test = torch.cat(all_preds).numpy()
y_te_np    = y_te_t.numpy()

print('=' * 60)
print('EVALUACIÓN EN TEST SET')
print('=' * 60)
print(f'Accuracy: {accuracy_score(y_te_np, preds_test)*100:.2f}%\n')
print(classification_report(y_te_np, preds_test, target_names=CLASS_NAMES))


In [ ]:
# ── Matrices de confusión ──
save_dir = data_path + f'results/EnfoquePorPartes/{CASO}/'
cm      = confusion_matrix(y_te_np, preds_test)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, data, title, fmt in [
    (axes[0], cm,      'Matriz de Confusión (conteos)',       lambda v: f'{v:,}'),
    (axes[1], cm_norm, 'Matriz de Confusión (normalizada)',   lambda v: f'{v*100:.1f}%'),
]:
    im = ax.imshow(data, cmap='Blues', vmin=0)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real'); ax.set_title(title)
    thresh = data.max() / 2
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, fmt(data[i, j]), ha='center', va='center',
                    color='white' if data[i, j] > thresh else 'black', fontsize=9)

plt.suptitle('ANN – Clasificación de Tipos de Enlace AS', fontsize=13, fontweight='bold')
plt.tight_layout()
save_dir = save_dir + f'results/EnfoquePorPartes/{CASO}/'
os.makedirs(save_dir, exist_ok=True)
plt.savefig(save_dir + f"results/EnfoquePorPartes/{CASO}/" +'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# ── Curvas de loss: sin batches vs con batches ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, tr, va, title in [
    # (axes[0], train_losses_nb, val_losses_nb, 'Sin Batches'),
    (axes[1], train_losses_b,  val_losses_b,  f'Con Batches (batch={ANN_BATCH})'),
]:
    ax.plot(tr, label='Train', linewidth=1.5)
    ax.plot(va, label='Val',   linewidth=1.5)
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Evolución del Loss – ANN Clasificador AS', fontsize=12)
plt.tight_layout()
plt.savefig(save_dir + 'loss_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Guardar modelos ──
# torch.save(model_nb.state_dict(), save_dir + f'ann_sin_batches_{embeddings_name}.pth')
torch.save(model_b.state_dict(),  save_dir + f'ann_con_batches_{embeddings_name}.pth')
print(f'Modelos guardados en {save_dir}')
